In [ ]:
!pip install -q datasets transformers torch scikit-learn pandas matplotlib fastapi uvicorn requests

In [ ]:
import random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Make results reproducible
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Select the available processing device
device = 0 if torch.cuda.is_available() else -1

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("Processing device:", "GPU" if device == 0 else "CPU")

In [ ]:
# Load the Yelp customer reviews dataset from Hugging Face
dataset = load_dataset("Yelp/yelp_review_full")

print(dataset)

In [ ]:
# Display sample reviews and their original labels
for i in range(3):
    print("Label:", dataset["test"][i]["label"])
    print("Review:", dataset["test"][i]["text"])
    print("-" * 80)

In [ ]:
# Select 100 test reviews and convert their star ratings into sentiment labels
test_sample = dataset["test"].shuffle(seed=42).select(range(100))

df = pd.DataFrame(test_sample)

def convert_sentiment(label):
    if label <= 1:
        return "negative"
    elif label == 2:
        return "neutral"
    else:
        return "positive"

df["stars"] = df["label"] + 1
df["sentiment"] = df["label"].apply(convert_sentiment)

print("Number of reviews:", len(df))
print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

In [ ]:
# Load a pretrained BERT model for customer review sentiment analysis
sentiment_model = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=device
)

print("Sentiment model loaded successfully.")

In [ ]:
# Predict star ratings and convert them into three sentiment categories
predictions = sentiment_model(
    df["text"].tolist(),
    batch_size=16,
    truncation=True
)

df["predicted_stars"] = [
    int(prediction["label"].split()[0])
    for prediction in predictions
]

df["predicted_sentiment"] = df["predicted_stars"].apply(
    lambda stars: "negative" if stars <= 2
    else "neutral" if stars == 3
    else "positive"
)

print("Sentiment predictions completed.")
print(df[["stars", "sentiment", "predicted_stars", "predicted_sentiment"]].head(10))

In [ ]:
# Evaluate the model by comparing predicted and actual sentiment labels
actual = df["sentiment"]
predicted = df["predicted_sentiment"]

accuracy = accuracy_score(actual, predicted)

print(f"Accuracy: {accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(actual, predicted, zero_division=0))

In [ ]:
# Visualise correct predictions and misclassifications for each sentiment
import seaborn as sns
import matplotlib.pyplot as plt

labels = ["negative", "neutral", "positive"]

cm = confusion_matrix(actual, predicted, labels=labels)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)

plt.xlabel("Predicted sentiment")
plt.ylabel("Actual sentiment")
plt.title("Yelp Review Sentiment Confusion Matrix")
plt.show()

In [ ]:
# Define common topics that may appear in Yelp customer reviews
topics = [
    "food quality",
    "customer service",
    "pricing and value",
    "cleanliness",
    "waiting time",
    "atmosphere and environment",
    "staff behaviour",
    "product quality"
]

print("Topics defined:")
for topic in topics:
    print("-", topic)

In [ ]:
# Load a pretrained model for identifying review topics without task-specific training
topic_model = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)

print("Topic detection model loaded successfully.")

In [ ]:
# Identify the most relevant topics in the first five customer reviews
topic_results = []

for review in df["text"].head(5):
    result = topic_model(
        review[:1024],
        candidate_labels=topics,
        multi_label=True
    )

    topic_results.append({
        "review": review[:150] + "...",
        "topics": result["labels"][:3],
        "scores": [round(score, 3) for score in result["scores"][:3]]
    })

topic_df = pd.DataFrame(topic_results)

print(topic_df.to_string(index=False))

In [ ]:
# Identify the three highest-scoring topics for each customer review
all_topic_results = []

for review in df["text"]:
    result = topic_model(
        review[:1024],
        candidate_labels=topics,
        multi_label=True
    )

    all_topic_results.append(", ".join(result["labels"][:3]))

df["detected_topics"] = all_topic_results

print("Topic detection completed.")
print(df[["sentiment", "detected_topics"]].head(10).to_string(index=False))

In [ ]:
# Load a pretrained model for summarising customer reviews
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=device
)

print("Summarisation model loaded successfully.")

In [ ]:
# Install a Transformers version that supports the summarization pipeline
!pip install -q "transformers==4.57.1"

In [ ]:
# Generate concise summaries while skipping very short reviews
summary_results = []

for review in df["text"].head(5):
    if len(review.split()) < 30:
        summary_results.append(review)
        continue

    result = summarizer(
        review,
        max_length=60,
        min_length=10,
        do_sample=False,
        truncation=True
    )

    summary_results.append(result[0]["summary_text"])

df["summary"] = pd.Series(summary_results, index=df.index[:5])

for i in range(5):
    print("Original review:", df.iloc[i]["text"][:200], "...")
    print("Summary:", df.iloc[i]["summary"])
    print("-" * 80)

In [ ]:
# Generate summaries using a maximum length based on each review's length
def generate_summary(review):
    words = len(review.split())

    if words < 30:
        return review

    max_summary_length = min(60, max(10, words // 2))

    result = summarizer(
        review,
        max_length=max_summary_length,
        min_length=min(10, max_summary_length - 1),
        do_sample=False,
        truncation=True
    )

    return result[0]["summary_text"]


df["summary"] = df["text"].apply(generate_summary)

print("Summarisation completed.")
print("Total summaries:", df["summary"].notna().sum())

In [ ]:
# Combine sentiment, detected topics, and summaries for review
results_df = df[
    ["text", "sentiment", "predicted_sentiment", "detected_topics", "summary"]
].copy()

print("Analysis results:")
display(results_df.head(10))

In [ ]:
# Import libraries needed to build and test our REST API
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List
import nest_asyncio
import uvicorn
import requests

nest_asyncio.apply()

print("API dependencies imported successfully.")

In [ ]:
# Create the FastAPI application
app = FastAPI(
    title="Yelp Review Analysis API",
    description="An NLP API for analysing customer reviews.",
    version="1.0.0"
)

print("FastAPI application created successfully.")

In [ ]:
# Define and validate incoming review requests
from typing import Annotated
from pydantic import StringConstraints

class ReviewRequest(BaseModel):
    text: Annotated[
        str,
        StringConstraints(
            strip_whitespace=True,
            min_length=1,
            max_length=5000
        )
    ]

In [ ]:
# Test review request validation
test_inputs = [
    {"text": "A good restaurant with excellent service."},
    {"text": ""},
    {"text": "     "},
    {"text": "A" * 5001}
]

for item in test_inputs:
    try:
        ReviewRequest(**item)
        print("Accepted:", repr(item["text"][:30]))
    except Exception as error:
        print("Rejected:", error.errors()[0]["msg"])

In [ ]:
# Create an endpoint to check API availability
@app.get("/health")
def health_check():
    return {
        "status": "healthy",
        "message": "Yelp Review Analysis API is running"
    }

In [ ]:
# Create an endpoint for customer review sentiment analysis
@app.post("/sentiment")
def analyse_sentiment(request: ReviewRequest):
    prediction = sentiment_model(
        request.text,
        truncation=True
    )[0]

    predicted_stars = int(prediction["label"].split()[0])

    if predicted_stars <= 2:
        sentiment = "negative"
    elif predicted_stars == 3:
        sentiment = "neutral"
    else:
        sentiment = "positive"

    return {
        "predicted_stars": predicted_stars,
        "sentiment": sentiment,
        "confidence": round(prediction["score"], 4)
    }

In [ ]:
# Create an endpoint for identifying customer review topics
@app.post("/topics")
def analyse_topics(request: ReviewRequest):
    result = topic_model(
        request.text[:1024],
        candidate_labels=topics,
        multi_label=True
    )

    return {
        "topics": [
            {
                "name": label,
                "confidence": round(score, 4)
            }
            for label, score in zip(
                result["labels"][:3],
                result["scores"][:3]
            )
        ]
    }

In [ ]:
# Endpoint for summarising customer reviews
@app.post("/summarize")
def summarize_review(request: ReviewRequest):
    summary = generate_summary(request.text)

    return {
        "summary": summary
    }

In [ ]:
# Endpoint for analysing a customer review using all three NLP techniques
@app.post("/analyse")
def analyse_review(request: ReviewRequest):
    sentiment_result = sentiment_model(
        request.text,
        truncation=True
    )[0]

    predicted_stars = int(sentiment_result["label"].split()[0])

    if predicted_stars <= 2:
        sentiment = "negative"
    elif predicted_stars == 3:
        sentiment = "neutral"
    else:
        sentiment = "positive"

    topic_result = topic_model(
        request.text[:1024],
        candidate_labels=topics,
        multi_label=True
    )

    summary = generate_summary(request.text)

    return {
        "sentiment": {
            "label": sentiment,
            "predicted_stars": predicted_stars,
            "confidence": round(sentiment_result["score"], 4)
        },
        "topics": [
            {
                "name": label,
                "confidence": round(score, 4)
            }
            for label, score in zip(
                topic_result["labels"][:3],
                topic_result["scores"][:3]
            )
        ],
        "summary": summary
    }

In [ ]:
# Start the API server
import threading

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

In [ ]:
# Test the API health endpoint
response = requests.get("http://127.0.0.1:8000/health")

print("Status code:", response.status_code)
print("Response:", response.json())

In [ ]:
# Test the sentiment analysis endpoint
sample_review = df.iloc[0]["text"]

response = requests.post(
    "http://127.0.0.1:8000/sentiment",
    json={"text": sample_review}
)

print("Status code:", response.status_code)
print("Response:", response.json())

In [ ]:
# Test the topic detection endpoint
response = requests.post(
    "http://127.0.0.1:8000/topics",
    json={"text": sample_review}
)

print("Status code:", response.status_code)
print("Response:", response.json())

In [ ]:
# Test the review summarisation endpoint
response = requests.post(
    "http://127.0.0.1:8000/summarize",
    json={"text": sample_review}
)

print("Status code:", response.status_code)
print("Response:", response.json())

In [ ]:
# Test the combined review analysis endpoint
response = requests.post(
    "http://127.0.0.1:8000/analyse",
    json={"text": sample_review}
)

print("Status code:", response.status_code)
print("Response:", response.json())

In [ ]:
# Test API validation with invalid review requests
invalid_inputs = [
    {"text": ""},
    {"text": "     "},
    {"text": "A" * 5001}
]

for item in invalid_inputs:
    response = requests.post(
        "http://127.0.0.1:8000/sentiment",
        json=item
    )

    print("Status code:", response.status_code)
    print("Error:", response.json()["detail"][0]["msg"])
    print("-" * 50)

In [ ]:
# Test the combined analysis endpoint on 20 Yelp reviews
test_results = []

for i in range(20):
    review = df.iloc[i]["text"]

    response = requests.post(
        "http://127.0.0.1:8000/analyse",
        json={"text": review}
    )

    test_results.append({
        "review_number": i + 1,
        "status_code": response.status_code,
        "successful": response.status_code == 200
    })

api_test_df = pd.DataFrame(test_results)

print("Total reviews tested:", len(api_test_df))
print("Successful requests:", api_test_df["successful"].sum())
print("Failed requests:", (~api_test_df["successful"]).sum())

In [ ]:
# Display the results of the API tests
display(api_test_df)

In [ ]:
# Summarise the sentiment model evaluation
evaluation_results = {
    "Evaluation dataset size": len(df),
    "Sentiment accuracy": f"{accuracy:.2%}",
    "Successful API requests": int(api_test_df["successful"].sum()),
    "Failed API requests": int((~api_test_df["successful"]).sum())
}

for metric, value in evaluation_results.items():
    print(f"{metric}: {value}")

In [ ]:
# Export the NLP analysis results to a CSV file
results_df.to_csv("yelp_analysis_results.csv", index=False)

print("Analysis results exported successfully.")
print("File: yelp_analysis_results.csv")

In [ ]:
# Download the analysis results to your computer
from google.colab import files

files.download("yelp_analysis_results.csv")

In [ ]:
# Summarise the topics identified across the Yelp reviews
topic_counts = (
    df["detected_topics"]
    .str.split(", ")
    .explode()
    .value_counts()
)

print("Topic distribution across the 100 reviews:")
display(topic_counts.to_frame(name="Number of reviews"))

In [ ]:
# Inspect sample topic and summary results to assess their quality
for i in range(5):
    row = df.iloc[i]
    print(f"Review {i + 1} ({row['stars']} stars):")
    print(row["text"][:400], "...")
    print("Detected topics:", row["detected_topics"])
    print("Summary:", row["summary"])
    print("-" * 80)

# NLP Application — Yelp Customer Review Analysis

## 1. Project Overview
This project implements an NLP-based system for analysing Yelp customer reviews. It uses pretrained transformer models to perform sentiment analysis, topic detection, and text summarisation.

## 2. Dataset
- **Source:** Yelp Review Full dataset from Hugging Face.
- **Evaluation sample:** 100 reviews.
- **Sentiment labels:** Negative, neutral, and positive, derived from the original 1–5 star ratings.

## 3. System Architecture
The application uses:
- **Python and Pandas:** Data processing and analysis.
- **Hugging Face Transformers:** Pretrained NLP models.
- **BERT multilingual sentiment model:** Sentiment classification and star-rating prediction.
- **BART-large-MNLI:** Zero-shot topic detection.
- **BART-large-CNN:** Review summarisation.
- **FastAPI:** REST API exposing the NLP functionality.
- **Uvicorn:** API server.

## 4. Implemented Features
1. **Sentiment analysis** — classifies reviews as positive, negative, or neutral.
2. **Topic detection** — identifies relevant subjects such as customer service, food quality, pricing, and cleanliness.
3. **Text summarisation** — generates shorter versions of customer reviews.

## 5. Evaluation Results
- **Sentiment accuracy:** 77% on 100 reviews.
- **API testing:** 20 out of 20 successful requests.
- **Most frequently top-ranked topics (top 3 per review):**
  - Customer service: 55
  - Atmosphere and environment: 54
  - Food quality: 52
  - Product quality: 43

## 6. Key Findings
The sentiment model achieved 77% accuracy on the 100-review evaluation sample, with neutral reviews presenting greater classification difficulty. Manual inspection of sample outputs showed that topic detection identified relevant subjects in several reviews, while summarisation generally retained key information from longer reviews. Short reviews were returned unchanged, and some topic assignments were less clearly supported by the review text, highlighting opportunities for improving output quality.

## 7. Limitations
- Sentiment labels were derived from star ratings rather than manually annotated sentiment labels.
- Topic detection uses predefined candidate topics, so topics outside the list may not be identified.
- Topic confidence scores represent model relevance scores, not verified probabilities.
- Summaries may occasionally omit details or inaccurately represent parts of a review.

## 8. Conclusion
The project demonstrates how pretrained transformer models can be combined to build a practical customer review analysis system. The FastAPI interface makes the NLP capabilities accessible through REST endpoints, while the evaluation provides an initial assessment of model performance and API functionality.